# DecodeLabs — Project 1: Rule-Based AI Chatbot

### Objective
Build a simple **rule-based chatbot** that responds to predefined user inputs using basic programming logic.

The chatbot should simulate a simple conversation between a human and a machine by:

1.   Receiving user input
2.   Processing the text
3.   Matching the input with predefined responses
4.   Returning an appropriate reply

This project introduces the fundamental concepts of Artificial Intelligence through control flow and decision-making logic rather than machine learning using:
- **Logic Engine**: The chatbot must use: if-else statements or A dictionary-based response system

  => The Logic Engine is responsible for:
  1. Understanding what the user typed
  2. Matching the input with known commands
  3. Selecting the correct response.
- **Heartbeat** (Continuous Loop): The chatbot should continue running until the user decides to stop it.
  
  => Using 'while True'. The loop allows:
  1. Continuous interaction
  2. Multiple conversations
  3. Real-time chatbot behavior

Without the loop, the chatbot would stop after one message.
- **Sanitizer** : Users may type inputs in different formats such as:
```
HELLO
 hello
Hello
```
  => To make the chatbot smarter, inputs should be cleaned before processing.

Sanitization includes:
1. Removing extra spaces
2. Converting text to lowercase
- **Kill Switch** (Exit Command) : The chatbot must provide a clean way to stop execution.
Example exit commands:
```
exit
quit
bye ```


## Input Sanitization

Raw user input is messy ("  HeLLo ", "HELLO", "hello").  
The **Sanitizer** normalizes everything before it hits the logic engine.

In [6]:
def sanitize(raw_input: str) -> str:
    """
    Normalize raw user text:
      1. .strip() : remove leading/trailing whitespace
      2. .lower() : convert to lowercase for case-insensitive matching
    """
    return raw_input.strip().lower()

# Test the sanitizer
test_inputs = ['  HeLLo  ', 'HELLO', 'hello', '  BYE ', '   HELP   ']

print("Sanitization:")
for raw in test_inputs:
    clean = sanitize(raw)
    print(f"  Raw: {repr(raw):20s}  →  Clean: {repr(clean)}")

Sanitization:
  Raw: '  HeLLo  '           →  Clean: 'hello'
  Raw: 'HELLO'               →  Clean: 'hello'
  Raw: 'hello'               →  Clean: 'hello'
  Raw: '  BYE '              →  Clean: 'bye'
  Raw: '   HELP   '          →  Clean: 'help'


## The Knowledge Base

Your chatbot's "brain" — a dictionary of **intent → response** mappings.  
The spec requires **5+ intents minimum**. This implementation ships with 12.

In [7]:
KNOWLEDGE_BASE = {
    # Greetings
    'hello'        : 'Hello! Welcome to DecoBot. How can I help you today?',
    'hi'           : 'Hi there! Great to see you. What can I do for you?',
    'hey'          : 'Hey! DecoBot is online and ready.',

    # Farewells
    'bye'          : 'Goodbye! See you in the next session.',
    'goodbye'      : 'Farewell, engineer. Keep building!',

    # Identity
    'name'         : 'My name is DecoBot — your rule-based AI companion at DecodeLabs.',
    'who are you'  : 'I am DecoBot, Project 1 of the DecodeLabs AI Industrial Training Kit.',

    # Capabilities
    'help'         : ' I can answer questions about DecodeLabs, AI concepts, and Project 1. Try: hello, name, joke, time, project.',
    'what can you do': ' I respond to predefined rules. Ask me: hello, help, name, joke, time, or project.',

    # Fun
    'joke'         : 'Why do programmers prefer dark mode? Because light attracts bugs!',

    # Utility
    'time'         : f' Server time is handled by the host. In a real system I would call datetime.now().',

    # Project info
    'project'      : ' You are working on Project 1: Rule-Based AI Chatbot. Complete it to unlock Week 2 projects!',
}

FALLBACK_RESPONSE = " I don't understand that input. Type 'help' to see what I can do."

print(f"Knowledge Base loaded — {len(KNOWLEDGE_BASE)} intents registered.")
print("\nAvailable commands:")
for key in KNOWLEDGE_BASE:
    print(f"  → '{key}'")

Knowledge Base loaded — 12 intents registered.

Available commands:
  → 'hello'
  → 'hi'
  → 'hey'
  → 'bye'
  → 'goodbye'
  → 'name'
  → 'who are you'
  → 'help'
  → 'what can you do'
  → 'joke'
  → 'time'
  → 'project'


## The Heartbeat (Infinite Loop)

The chatbot **stays alive** inside a `while True` loop.  
The only way out is the **Kill Command** — typing `exit` or `quit`.

In [8]:
EXIT_COMMANDS = {'exit', 'quit', 'q', 'stop', 'end'}

def run_chatbot():
    """Main chatbot loop — the complete rule-based AI system."""

    # Startup banner
    print("=" * 55)
    print("  🤖  DecoBot — Rule-Based AI Chatbot")
    print("=" * 55)
    print("  Type 'help' for available commands.")
    print("  Type 'exit' to quit.")

    # Infinite loop
    while True:

        # PHASE 1 _ INPUT & SANITIZATION
        raw_input  = input("You: ")
        clean_input = sanitize(raw_input)   # .strip().lower()

        # EXIT STRATEGY — Kill Command
        if clean_input in EXIT_COMMANDS:
            print("DecoBot: Shutting down. Session ended. Goodbye, engineer")
            print("-" * 55)
            break  # exits the while True loop cleanly

        # Guard against empty input
        if not clean_input:
            print("DecoBot: Please type something")
            continue

        # PHASE 2 — PROCESS: O(1) dictionary lookup + fallback
        reply = KNOWLEDGE_BASE.get(clean_input, FALLBACK_RESPONSE)

        # PHASE 3 — OUTPUT
        print(f"DecoBot: {reply}")
        print()  # blank line for readability


# ── Launch the chatbot ──────────────────────────────────────
run_chatbot()

  🤖  DecoBot — Rule-Based AI Chatbot
  Type 'help' for available commands.
  Type 'exit' to quit.
You: hi
DecoBot: Hi there! Great to see you. What can I do for you?

You: hello
DecoBot: Hello! Welcome to DecoBot. How can I help you today?

You: exit
DecoBot: Shutting down. Session ended. Goodbye, engineer
-------------------------------------------------------


##  Challenge Extensions


In [9]:
# EXTENSION 1 — PARTIAL / FUZZY KEYWORD MATCHING
#
# Problem: the base chatbot only matches EXACT inputs.
#   "hello there" → fallback  (even though it contains 'hello')
#
# Solution: check if any keyword from the knowledge base
#           appears INSIDE the user's message.

def get_reply_with_keyword_match(clean_input: str) -> str:
    """
    Step 1: Try exact match (O(1) — fastest)
    Step 2: Try keyword-in-message scan  (O(n) — only as fallback)
    Step 3: Return default fallback
    """
    # --- Exact match (O(1)) ---
    if clean_input in KNOWLEDGE_BASE:
        return KNOWLEDGE_BASE[clean_input]

    # --- Keyword scan (O(n), only when exact fails) ---
    for keyword, response in KNOWLEDGE_BASE.items():
        if keyword in clean_input:  # e.g. 'hello' in 'hello there'
            return f"(keyword match) {response}"

    # --- Fallback ---
    return FALLBACK_RESPONSE


# Test fuzzy matching
test_phrases = [
    'hello there friend',
    'can you tell me a joke please',
    'what is the project about',
    'random gibberish xyz',
]

print("Extension 1 — Partial Keyword Matching Demo")
print("-" * 50)
for phrase in test_phrases:
    reply = get_reply_with_keyword_match(phrase.lower())
    print(f"  Input : '{phrase}'")
    print(f"  Reply : {reply}")
    print()

Extension 1 — Partial Keyword Matching Demo
--------------------------------------------------
  Input : 'hello there friend'
  Reply : (keyword match) Hello! Welcome to DecoBot. How can I help you today?

  Input : 'can you tell me a joke please'
  Reply : (keyword match) Why do programmers prefer dark mode? Because light attracts bugs!

  Input : 'what is the project about'
  Reply : (keyword match)  You are working on Project 1: Rule-Based AI Chatbot. Complete it to unlock Week 2 projects!

  Input : 'random gibberish xyz'
  Reply :  I don't understand that input. Type 'help' to see what I can do.



In [10]:
# EXTENSION 2 — CONTEXT / STATE TRACKING
#
# A stateless bot forgets everything after each turn.
# Add a simple session memory to track:
#   - How many messages the user has sent
#   - Whether the user has introduced themselves

def run_chatbot_with_memory():
    """Chatbot with session state tracking."""

    # --- Session state ---
    state = {
        'username'      : None,
        'message_count' : 0,
    }

    print("=" * 55)
    print("DecoBot v2 — With Memory")
    print("-" * 55)
    print("  Special commands: 'my name is <name>', 'who am i', 'count'")
    print("  Type 'exit' to quit.")
    print("-" * 55)

    while True:

        raw_input   = input("You: ")
        clean_input = sanitize(raw_input)
        state['message_count'] += 1

        # Kill command
        if clean_input in EXIT_COMMANDS:
            print(f"DecoBot: Session ended after {state['message_count']} messages. Bye!")
            break

        if not clean_input:
            continue

        # --- Context-aware rules (checked BEFORE knowledge base) ---

        # Rule: user introduces themselves
        if clean_input.startswith('my name is '):
            name = clean_input.replace('my name is ', '').strip().title()
            state['username'] = name
            print(f"DecoBot: Nice to meet you, {name}! I'll remember that.\n")
            continue

        # Rule: user asks who they are
        if clean_input == 'who am i':
            if state['username']:
                print(f"DecoBot: You told me your name is {state['username']}.\n")
            else:
                print("DecoBot: I don't know yet. Try: 'my name is <your name>'\n")
            continue

        # Rule: message count
        if clean_input == 'count':
            print(f"DecoBot: You've sent {state['message_count']} message(s) this session.\n")
            continue

        # --- Standard knowledge base lookup ---
        reply = KNOWLEDGE_BASE.get(clean_input, FALLBACK_RESPONSE)

        # Personalize if username is known
        if state['username'] and clean_input in ('hello', 'hi', 'hey'):
            reply = f" Hello again, {state['username']}! Good to see you."

        print(f"DecoBot: {reply}\n")


# ── Launch v2 ───────────────────────────────────────────────
run_chatbot_with_memory()

DecoBot v2 — With Memory
-------------------------------------------------------
  Special commands: 'my name is <name>', 'who am i', 'count'
  Type 'exit' to quit.
-------------------------------------------------------
You: hello
DecoBot: Hello! Welcome to DecoBot. How can I help you today?

You: exit
DecoBot: Session ended after 2 messages. Bye!


## Architecture Recap
The system takes user input, cleans it (trim and lowercase), then checks it against a predefined KNOWLEDGE_BASE. If a match is found, it returns the stored response; otherwise, it gives a fallback message. The output is displayed with print(). This process runs in a continuous loop (while True) until an exit command is entered, which stops the program.